# supervised_baseline

**supervised model**  
CIFAR-10 → Data Augmentation → ResNet-18 → 512→10 classifier → Cross Entropy → Test Accuracy

## step1:Import + 基本設定

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

import matplotlib.pyplot as plt
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## step2:準備CIFAR-10 Dataset

In [3]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

c:\Users\user\anaconda3\envs\summer\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000
Test: 10000


## step3:建立 Supervised ResNet-18

In [4]:
class SupervisedResNet18(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = resnet18(weights=None)

        # CIFAR-10 修改
        self.model.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.model.maxpool = nn.Identity()

        # 直接分類 CIFAR-10 10 classes
        self.model.fc = nn.Linear(512, 10)

    def forward(self, x):
        return self.model(x)


model = SupervisedResNet18().to(device)

print(model.model.fc)

Linear(in_features=512, out_features=10, bias=True)


## step4:測輸出維度

In [5]:
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input:", images.shape)
print("Output:", outputs.shape)

Input: torch.Size([256, 3, 32, 32])
Output: torch.Size([256, 10])


## step5:設定 Loss 與 Optimizer

In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-6
)

print("Loss: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning rate: 3e-4")
print("Weight decay: 1e-6")

Loss: CrossEntropyLoss
Optimizer: Adam
Learning rate: 3e-4
Weight decay: 1e-6


## step6:跑 1 epoch 測試

In [7]:
model.train()

total_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for images, labels in train_loader:

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)

    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    correct += (predictions == labels).sum().item()
    total += labels.size(0)

avg_loss = total_loss / len(train_loader)
train_acc = 100 * correct / total

elapsed = time.time() - start_time

print("===== Epoch 1 Test =====")
print(f"Loss: {avg_loss:.4f}")
print(f"Training Accuracy: {train_acc:.2f}%")
print(f"Time: {elapsed:.1f} seconds")

===== Epoch 1 Test =====
Loss: 1.4429
Training Accuracy: 47.16%
Time: 28.8 seconds


## step7:正式訓練 200 Epochs

In [8]:
# =====================================
# 重新初始化模型，確保從頭開始
# =====================================
model = SupervisedResNet18().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-6
)

num_epochs = 200

train_loss_history = []
train_acc_history = []
test_acc_history = []

training_start = time.time()

for epoch in range(1, num_epochs + 1):

    # =====================
    # Training
    # =====================
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    epoch_start = time.time()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = 100.0 * correct / total

    # =====================
    # Test
    # =====================
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    test_acc = 100.0 * correct / total

    # 儲存結果
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)

    epoch_time = time.time() - epoch_start

    print(
        f"Epoch [{epoch:3d}/{num_epochs}] "
        f"| Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.2f}% "
        f"| Test Acc: {test_acc:.2f}% "
        f"| Time: {epoch_time:.1f}s"
    )

# =====================================
# 儲存模型
# =====================================
torch.save(
    model.state_dict(),
    "supervised_baseline_final.pth"
)

total_time = time.time() - training_start

print("\n===== Supervised Training Finished =====")
print(f"Total Time: {total_time / 3600:.2f} hours")
print(f"Final Train Accuracy: {train_acc_history[-1]:.2f}%")
print(f"Final Test Accuracy: {test_acc_history[-1]:.2f}%")
print(f"Best Test Accuracy: {max(test_acc_history):.2f}%")

Epoch [  1/200] | Loss: 1.4183 | Train Acc: 47.86% | Test Acc: 49.99% | Time: 31.4s
Epoch [  2/200] | Loss: 0.9795 | Train Acc: 64.93% | Test Acc: 65.93% | Time: 31.7s
Epoch [  3/200] | Loss: 0.7779 | Train Acc: 72.57% | Test Acc: 72.99% | Time: 32.2s
Epoch [  4/200] | Loss: 0.6541 | Train Acc: 77.08% | Test Acc: 78.09% | Time: 32.2s
Epoch [  5/200] | Loss: 0.5789 | Train Acc: 79.78% | Test Acc: 74.67% | Time: 30.1s
Epoch [  6/200] | Loss: 0.5129 | Train Acc: 82.05% | Test Acc: 79.40% | Time: 29.4s
Epoch [  7/200] | Loss: 0.4693 | Train Acc: 83.64% | Test Acc: 78.24% | Time: 29.5s
Epoch [  8/200] | Loss: 0.4358 | Train Acc: 84.74% | Test Acc: 82.68% | Time: 29.4s
Epoch [  9/200] | Loss: 0.3972 | Train Acc: 86.26% | Test Acc: 83.00% | Time: 29.4s
Epoch [ 10/200] | Loss: 0.3638 | Train Acc: 87.25% | Test Acc: 80.24% | Time: 29.5s
Epoch [ 11/200] | Loss: 0.3433 | Train Acc: 88.03% | Test Acc: 84.88% | Time: 29.4s
Epoch [ 12/200] | Loss: 0.3196 | Train Acc: 88.95% | Test Acc: 84.60% | Time